# 09 Robustness LMS 30x80


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import hashlib

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# Find paper root from the standard Drive path or from the local package.
_candidates = [
    Path(os.environ.get('CARE_KM_PAPER_ROOT', '')) if os.environ.get('CARE_KM_PAPER_ROOT') else None,
    _ROOT_REVISED,
    _ROOT_REVISED,
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
]
PAPER_ROOT = next((p.resolve() for p in _candidates if p is not None and (p/'LIB').is_dir() and (p/'DATA').is_dir()), None)
if PAPER_ROOT is None:
    raise FileNotFoundError('Root REVISED tidak ditemukan.')
LIB_DIR = PAPER_ROOT / 'LIB'
DATA_DIR = PAPER_ROOT / 'DATA'
OUT_DIR = stage_dir('09_Robustness_LMS_30x80')
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))

import experiment_lib as el
import care_km as ck

CARE_CONFIG = dict(el.FROZEN_CARE_CONFIG)
K = el.K
METHODS = list(el.METHODS)
print('PAPER_ROOT:', PAPER_ROOT)
print('CARE config:', CARE_CONFIG)


df, X_all, FEATURES = el.load_lms_dataset(PAPER_ROOT)
REPS = 30
FRACTION = 0.80
BASE_SEED = 6161600
CHECKPOINT = artifact_path('06_robustness_raw.csv','09_Robustness_LMS_30x80')

if CHECKPOINT.exists():
    raw = pd.read_csv(CHECKPOINT)
else:
    raw = pd.DataFrame()

for rep in range(REPS):
    rng = np.random.default_rng(BASE_SEED + rep)
    idx = np.sort(rng.choice(len(X_all), size=int(len(X_all)*FRACTION), replace=False))
    X = X_all[idx]
    sample_hash = el.sample_index_hash(idx)
    record_resample('09_Robustness_LMS_30x80','LMS',rep,idx,sample_hash)

    done = set()
    if len(raw):
        done = set(raw.loc[pd.to_numeric(raw['rep']) == rep, 'method'].astype(str))

    new_rows = []
    for method in METHODS:
        if method in done:
            continue
        row, _, _ = ck.run_method(
            method, X, None, K,
            seed=BASE_SEED + rep,
            care_config=CARE_CONFIG,
            si_sample_size=el.SI_SAMPLE_SIZE,
            si_seed=BASE_SEED + rep,
            max_iter=el.MAX_ITER,
        )
        row.update({'rep': rep, 'sample_hash': sample_hash, 'sample_n': len(idx)})
        new_rows.append(row)

    if new_rows:
        raw = pd.concat([raw, pd.DataFrame(new_rows)], ignore_index=True)
        el.save_csv(raw, CHECKPOINT)

# Integrity: 30 reps x 7 methods, exactly one shared sample hash per rep.
counts = raw.groupby('rep')['method'].nunique()
if len(counts) != REPS or not (counts == len(METHODS)).all():
    raise RuntimeError('Robustness checkpoint belum lengkap 30 x 7.')
if not (raw.groupby('rep')['sample_hash'].nunique() == 1).all():
    raise RuntimeError('Matched-sample integrity gagal.')

summary = raw.groupby('method').agg(
    n=('rep','size'),
    initial_cost_mean=('initial_cost','mean'), initial_cost_sd=('initial_cost','std'),
    final_cost_mean=('final_cost','mean'), final_cost_sd=('final_cost','std'),
    dbi_mean=('dbi','mean'), dbi_sd=('dbi','std'),
    si_mean=('si','mean'), si_sd=('si','std'),
    init_sec_mean=('init_sec','mean'), init_sec_sd=('init_sec','std'),
    cluster_sec_mean=('cluster_sec','mean'), cluster_sec_sd=('cluster_sec','std'),
    total_sec_mean=('total_sec','mean'), total_sec_sd=('total_sec','std'),
    iterations_mean=('iterations','mean'), iterations_sd=('iterations','std'),
).reset_index()
summary['rank_final_cost'] = summary['final_cost_mean'].rank(ascending=True)
summary['rank_dbi'] = summary['dbi_mean'].rank(ascending=True)
summary['rank_si'] = summary['si_mean'].rank(ascending=False)
summary['rank_primary'] = summary[['rank_final_cost','rank_dbi','rank_si']].mean(axis=1)
summary = summary.sort_values(['rank_primary','final_cost_mean'])

el.save_csv(summary, artifact_path('06_robustness_summary.csv','09_Robustness_LMS_30x80'))
display(summary)
print('Matched resampling integrity: PASS')